#**Credit Risk Modelling — Retail Lending Portfolio**
##Project Type
Predictive Modelling (Classification) — Credit Risk Analytics
##Team Member
Sakshi Chore

##Github Link
https://github.com/sakshi200-code/credit-risk-rbi-project

##Project Summary
This project builds and validates a credit default risk model for a retail lending portfolio of 32,500+ records, with an observed default rate of 21.8%. The workflow covers data quality auditing, outlier cleaning, segment-level default rate analysis, interpretable feature engineering, logistic regression modelling, risk-band construction, and portfolio stress testing.

The model achieves a ROC-AUC of 0.86 and a KS statistic of 0.59, and separates borrowers into five risk bands where the observed default rate rises from 2.5% in the lowest-risk band to 68.9% in the highest-risk band — demonstrating strong risk-ranking power suitable for underwriting and portfolio monitoring use cases.

Python libraries such as Pandas, NumPy, and Scikit-learn are used for data cleaning, feature engineering, model training, and validation. Logistic regression was chosen deliberately for its interpretability, which matters in regulated lending contexts where model decisions must be explainable to underwriters, risk committees, and regulators.

## Project Objective

The objective of this project is to estimate credit default risk for retail loan applicants using borrower, loan, and credit-history variables — supporting underwriting decisions, portfolio monitoring, early warning systems, and regulatory-style risk discussions (RBI / banking interview context).

# Credit Risk Analyst Case Study

**End-to-end credit risk analysis on a retail lending portfolio.**

This notebook walks through the complete analyst workflow used to build and validate a
credit default risk model, from raw data to a stakeholder-ready case study report.
The reusable, production-style logic lives in `src/`; this notebook narrates the
workflow step by step for portfolio review and interview discussion.

**Workflow:**
1. Load and audit the lending dataset
2. Clean outliers and unrealistic values
3. Build exploratory default-rate summary tables
4. Train an interpretable credit risk model (logistic regression)
5. Validate discriminatory power (AUC, KS)
6. Convert scores into risk bands
7. Run portfolio stress scenarios
8. Export a full case study report

## 1. Setup

In [ ]:
# Standard library / third-party imports
from pathlib import Path

import numpy as np
import pandas as pd

# Project modules — the modular, production-style implementation
from src.data_loading import TARGET, load_credit_risk_dataset
from src.model_pipeline import train_and_evaluate
from src.reporting import write_interview_notes, write_model_summary
from src.stress_testing import run_stress_tests

In [ ]:
# Project paths — keep all inputs/outputs organized under one base directory
BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
REPORT_DIR = BASE_DIR / "reports"
TABLE_DIR = REPORT_DIR / "tables"

ZIP_PATH = DATA_DIR / "archive.zip"
EXTRACTED_CSV_PATH = DATA_DIR / "credit_risk_dataset.csv"

MODEL_SUMMARY_PATH = REPORT_DIR / "model_summary.md"
INTERVIEW_NOTES_PATH = REPORT_DIR / "interview_notes.md"
CASE_STUDY_PATH = REPORT_DIR / "full_case_study.md"


def ensure_directories() -> None:
    """Create the data/report/table folders if they don't already exist."""
    DATA_DIR.mkdir(exist_ok=True)
    REPORT_DIR.mkdir(exist_ok=True)
    TABLE_DIR.mkdir(exist_ok=True)

## 2. Data Quality Audit

Before any modelling, profile every column for missingness, cardinality, and type —
and summarize the numeric and categorical variables separately. This is the same
first pass any credit risk analyst would run on a new lending dataset.

In [ ]:
def basic_dataset_profile(data: pd.DataFrame) -> pd.DataFrame:
    """Per-column data quality profile: dtype, missingness, and cardinality."""
    rows = []
    for column in data.columns:
        series = data[column]
        rows.append(
            {
                "column": column,
                "dtype": str(series.dtype),
                "non_null": int(series.notna().sum()),
                "missing": int(series.isna().sum()),
                "missing_pct": round(float(series.isna().mean() * 100), 2),
                "unique_values": int(series.nunique(dropna=True)),
            }
        )
    return pd.DataFrame(rows)


def numeric_summary(data: pd.DataFrame) -> pd.DataFrame:
    """Descriptive statistics (incl. tail percentiles) for all numeric columns."""
    numeric = data.select_dtypes(include=["number"])
    summary = numeric.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    summary = summary.reset_index().rename(columns={"index": "feature"})
    return summary.round(4)


def categorical_summary(data: pd.DataFrame) -> pd.DataFrame:
    """Value counts and share of records for every categorical column."""
    categorical_columns = data.select_dtypes(include=["object", "string", "category"]).columns
    rows = []
    for column in categorical_columns:
        counts = data[column].value_counts(dropna=False)
        for value, count in counts.items():
            rows.append(
                {
                    "feature": column,
                    "value": str(value),
                    "records": int(count),
                    "share_pct": round(float(count / len(data) * 100), 2),
                }
            )
    return pd.DataFrame(rows)

## 3. Segment-Level Default Rate Analysis

A single model score isn't enough for a business conversation — analysts need to show
*where* risk concentrates. These functions break default rate down by categorical
segment (loan grade, purpose, home ownership) and by quantile bands of numeric
variables (income, interest rate, credit history length).

In [ ]:
def default_rate_by_category(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Default rate, record count, and default count for each category value."""
    grouped = (
        data.groupby(column, dropna=False)[TARGET]
        .agg(records="size", defaults="sum", default_rate="mean")
        .reset_index()
        .sort_values("default_rate", ascending=False)
    )
    grouped["default_rate"] = grouped["default_rate"].round(4)
    return grouped


def default_rate_by_numeric_bins(data: pd.DataFrame, column: str, bins: int = 5) -> pd.DataFrame:
    """Default rate across quantile bands (quintiles by default) of a numeric feature."""
    temporary = data[[column, TARGET]].dropna().copy()
    temporary[f"{column}_band"] = pd.qcut(temporary[column], q=bins, duplicates="drop")
    grouped = (
        temporary.groupby(f"{column}_band", observed=False)[TARGET]
        .agg(records="size", defaults="sum", default_rate="mean")
        .reset_index()
    )
    grouped[f"{column}_band"] = grouped[f"{column}_band"].astype(str)
    grouped["default_rate"] = grouped["default_rate"].round(4)
    return grouped

## 4. Feature Engineering

Four simple, business-interpretable features derived from the raw columns — a
loan-to-income ratio, and flags for high interest rate, short credit history, and
prior default on file. Each one has a clear risk rationale, which matters more in a
banking interview than model complexity.

In [ ]:
def create_derived_features(data: pd.DataFrame) -> pd.DataFrame:
    """Add interpretable risk-driver features on top of the raw columns."""
    enriched = data.copy()
    enriched["loan_to_income_ratio"] = enriched["loan_amnt"] / enriched["person_income"].replace(0, np.nan)
    enriched["high_interest_flag"] = (enriched["loan_int_rate"] >= enriched["loan_int_rate"].median()).astype(int)
    enriched["short_credit_history_flag"] = (enriched["cb_person_cred_hist_length"] <= 3).astype(int)
    enriched["prior_default_flag"] = (enriched["cb_person_default_on_file"] == "Y").astype(int)
    return enriched


def feature_relationship_tables(data: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Build every default-rate-by-segment table used in the case study report."""
    tables = {}

    # Categorical segments most relevant to underwriting decisions
    categorical_features = [
        "person_home_ownership",
        "loan_intent",
        "loan_grade",
        "cb_person_default_on_file",
    ]
    for feature in categorical_features:
        tables[f"default_by_{feature}"] = default_rate_by_category(data, feature)

    # Numeric drivers, viewed in quantile bands for easier business interpretation
    numeric_features = [
        "person_age",
        "person_income",
        "person_emp_length",
        "loan_amnt",
        "loan_int_rate",
        "loan_percent_income",
        "cb_person_cred_hist_length",
    ]
    for feature in numeric_features:
        tables[f"default_by_{feature}_band"] = default_rate_by_numeric_bins(data, feature)

    enriched = create_derived_features(data)
    tables["derived_feature_summary"] = enriched[
        [
            "loan_to_income_ratio",
            "high_interest_flag",
            "short_credit_history_flag",
            "prior_default_flag",
            TARGET,
        ]
    ].describe().T.reset_index().rename(columns={"index": "feature"}).round(4)

    return tables

## 5. Reporting Utilities

Helper functions to export every summary table to CSV and to render a DataFrame as a
clean Markdown table for the final report.

In [ ]:
def export_tables(tables: dict[str, pd.DataFrame]) -> None:
    """Write every summary table to its own CSV file under reports/tables/."""
    for name, table in tables.items():
        table.to_csv(TABLE_DIR / f"{name}.csv", index=False)


def markdown_table(frame: pd.DataFrame, max_rows: int | None = None) -> str:
    """Render a DataFrame as a GitHub-flavoured Markdown table."""
    display = frame.copy()
    if max_rows is not None:
        display = display.head(max_rows)

    for column in display.columns:
        if pd.api.types.is_float_dtype(display[column]):
            display[column] = display[column].map(lambda value: f"{value:.4f}")

    headers = [str(column) for column in display.columns]
    rows = display.astype(str).values.tolist()
    header_row = "| " + " | ".join(headers) + " |"
    separator_row = "| " + " | ".join(["---"] * len(headers)) + " |"
    body_rows = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header_row, separator_row, *body_rows])

## 6. Case Study Report

Assembles every table, metric, and stress-test result produced above into a single
narrative Markdown report (`reports/full_case_study.md`) — structured the way I'd
walk an interviewer through the project: business context, data audit, EDA, model
choice, validation, risk bands, key drivers, stress testing, and limitations.

In [ ]:
def write_full_case_study(
    audit: dict,
    profile: pd.DataFrame,
    numeric: pd.DataFrame,
    tables: dict[str, pd.DataFrame],
    model_results: dict,
    stress_results: pd.DataFrame,
) -> None:
    """Write the full narrative case study report to CASE_STUDY_PATH."""
    metrics = model_results["metrics"]
    coefficients = model_results["top_coefficients"]
    risk_bands = model_results["risk_bands"]

    content = f"""# Credit Risk Analyst Case Study

## 1. Business Context

The objective is to estimate credit default risk for loan applicants using borrower, loan, and credit-history variables. This type of analysis supports underwriting, portfolio monitoring, early warning systems, and regulatory-style risk discussions.

## 2. Dataset Overview

- Source records: {audit["source_rows"]:,}
- Records used after cleaning: {audit["model_rows"]:,}
- Removed records: {audit["removed_rows"]:,}
- Target column: `{audit["target_column"]}`
- Observed default rate: {audit["default_rate"]:.2%}

## 3. Data Quality Audit

{markdown_table(profile)}

Key cleaning choices:

- Removed unrealistic borrower ages outside 18 to 100.
- Removed unrealistic employment length above 60 years.
- Removed non-positive income or loan amount records.
- Imputed missing numeric values inside the modelling pipeline.

## 4. Numeric Variable Summary

{markdown_table(numeric, max_rows=12)}

## 5. Default Rate by Loan Grade

{markdown_table(tables["default_by_loan_grade"])}

## 6. Default Rate by Loan Purpose

{markdown_table(tables["default_by_loan_intent"])}

## 7. Default Rate by Home Ownership

{markdown_table(tables["default_by_person_home_ownership"])}

## 8. Default Rate by Loan Percent Income Band

{markdown_table(tables["default_by_loan_percent_income_band"])}

## 9. Model Choice

I used logistic regression because it is interpretable, stable, and easy to explain in banking interviews. The goal is not only prediction accuracy, but also understanding why borrowers are classified as higher or lower risk.

## 10. Model Validation

| Metric | Value |
| --- | ---: |
| ROC AUC | {metrics["auc"]:.4f} |
| KS Statistic | {metrics["ks"]:.4f} |
| Accuracy | {metrics["accuracy"]:.4f} |
| Precision | {metrics["precision"]:.4f} |
| Recall | {metrics["recall"]:.4f} |
| F1 Score | {metrics["f1"]:.4f} |

The AUC and KS values show that the model separates high-risk and low-risk borrowers well. Recall is important because missing risky borrowers can create credit losses.

## 11. Risk Bands

{markdown_table(risk_bands)}

Risk bands convert raw probabilities into a portfolio-monitoring view. The highest band should have a much higher default rate than the lowest band. This project shows clear rank ordering.

## 12. Top Model Drivers

{markdown_table(coefficients[["feature", "coefficient"]])}

Positive coefficients increase estimated risk. Negative coefficients reduce estimated risk. These drivers should be reviewed with business judgment before any production use.

## 13. Stress Testing

{markdown_table(stress_results)}

The stress test increases loan interest rate and loan percent of income to simulate weaker affordability and pricing conditions. The result shows how portfolio average PD changes under mild and severe assumptions.

## 14. RBI / Banking Interview Discussion

This project demonstrates the credit risk lifecycle:

- Define a default target.
- Audit and clean raw data.
- Study default rates across borrower and loan segments.
- Train an interpretable model.
- Validate ranking power using AUC and KS.
- Convert predictions into risk bands.
- Explain model drivers.
- Stress the portfolio under adverse assumptions.

## 15. Limitations

- The dataset does not include macroeconomic time series.
- The model does not include out-of-time validation because no date column is available.
- The project is educational and not a production credit decision system.
- Further work should include calibration plots, fairness checks, PSI monitoring, and challenger models.
"""
    CASE_STUDY_PATH.write_text(content, encoding="utf-8")

## 7. Run the Full Pipeline

Ties every step together: load data → audit → EDA tables → train & validate model →
stress test → export CSVs and the final case study report.

In [ ]:
def main() -> None:
    ensure_directories()

    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"Dataset not found: {ZIP_PATH}")

    # Step 1: load and audit the raw dataset
    data, audit = load_credit_risk_dataset(ZIP_PATH, EXTRACTED_CSV_PATH)

    # Step 2: profile data quality and summarize variables
    profile = basic_dataset_profile(data)
    numeric = numeric_summary(data)
    categorical = categorical_summary(data)
    relationship_tables = feature_relationship_tables(data)

    all_tables = {
        "dataset_profile": profile,
        "numeric_summary": numeric,
        "categorical_summary": categorical,
        **relationship_tables,
    }
    export_tables(all_tables)

    # Step 3: train and validate the credit risk model
    model_results = train_and_evaluate(data, target=TARGET)

    # Step 4: run portfolio stress scenarios on the fitted model
    stress_results = run_stress_tests(model_results["model"], model_results["feature_columns"], data)

    # Step 5: write out the model summary, interview notes, and full case study
    write_model_summary(MODEL_SUMMARY_PATH, model_results, stress_results, audit)
    write_interview_notes(INTERVIEW_NOTES_PATH)
    write_full_case_study(audit, profile, numeric, relationship_tables, model_results, stress_results)

    print("Credit risk analysis completed.")
    print(f"Model summary: {MODEL_SUMMARY_PATH}")
    print(f"Full case study: {CASE_STUDY_PATH}")
    print(f"Exported tables: {TABLE_DIR}")


if __name__ == "__main__":
    main()

# Executive Summary

## Project Objective

The objective of this project was to estimate credit default risk for retail loan applicants using borrower, loan, and credit-history variables, and to convert that risk into a form usable for underwriting and portfolio monitoring decisions.

---

## Key Findings

- The portfolio's observed default rate was 21.8% across 32,500+ retail lending records.
- Income, loan grade, interest rate, and credit history length were the primary features used to rank borrower risk.
- The logistic regression model achieved a ROC-AUC of 0.86 and a KS statistic of 0.59, indicating strong separation between defaulting and non-defaulting borrowers.
- Segmenting borrowers into five risk bands showed a clear, monotonic rise in default rate — from 2.5% in the lowest-risk band to 68.9% in the highest-risk band — confirming the model ranks risk consistently rather than just predicting a single average outcome.
- Portfolio stress testing, simulating higher interest rates and loan-to-income burdens, projected a 27% to 69% rise in average predicted default risk under mild-to-severe adverse scenarios.
- Logistic regression's interpretability made it possible to identify and explain individual risk drivers, which is important for underwriting decisions that need to be justified to risk committees and regulators.

---

## Business Recommendations

- Use the five-tier risk-band structure to differentiate underwriting terms (approval thresholds, pricing, collateral requirements) rather than relying on a single pass/fail cutoff.
- Prioritize enhanced review for applicants falling in the top two risk bands, where observed default rates are substantially higher than the portfolio average.
- Incorporate the stress-test results into capital and provisioning planning, since a mild-to-severe rate/affordability shock could raise expected defaults by up to ~69%.
- Revisit model coefficients periodically with business judgment before any production deployment, since coefficients reflect historical relationships that may shift with macroeconomic conditions.
- Extend future iterations with out-of-time validation, calibration checks, fairness testing, and population stability monitoring, as noted in the project's limitations.

# Conclusion

The analysis shows that an interpretable logistic regression model can rank-order retail lending risk effectively, achieving strong discriminatory power (AUC 0.86, KS 0.59) while remaining explainable to underwriting and risk teams. The resulting risk bands and stress-test outputs give a practical foundation for underwriting, monitoring, and capital-planning decisions. Before any production use, the model would benefit from out-of-time validation, calibration analysis, and fairness review — but as a portfolio risk-ranking tool, it demonstrates the core credit risk analytics lifecycle end to end.